# Slides / Report   
---

---
# Handson Verilog – Seven Segment Display demo


## FPGA implementation

### Steps
- Environment Check-Up
- Device Setup
- Module 
- Testbench (no need, as this demo direct implementation)
- Constraints 
- Create Vivado Project
- Run RTL Simulation (no need, as this demo direct implementation only)

In [54]:
# Environment Check-Up
! vivado -version
%load_ext vivado_magic

vivado v2025.1 (64-bit)
Tool Version Limit: 2025.05
SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.
The vivado_magic extension is already loaded. To reload it, use:
  %reload_ext vivado_magic


In [55]:
device_part  = "xc7a35tcpg236-1" # Do not touch
project_name = "A7E_ring_osc"
design_top   = "ring_osc"
sim_top      = "ring_osc_tb"
%vivado prj_clean

[INFO]: Removing build directory: /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc


## Module and Testbench

# Ring Oscillator Random Number Display — Cmod A7

All pins taken directly from `A7E_stopwatch` XDC — zero assumptions.

| Port | Pin | Location |
|---|---|---|
| `sysclk` | L17 | Onboard CMod A7 |
| `SW[0]` enable osc | V8 | Extension board |
| `SW[1]` reset | W7 | Extension board |
| `btn[0]` freeze display | A18 | Onboard CMod A7 |
| `btn[1]` unused | B18 | Onboard CMod A7 |
| `led[0]` osc running | A17 | Onboard CMod A7 |
| `led[1]` heartbeat | C16 | Onboard CMod A7 |
| `HEX[5:0]` digit selects | N2,N1,L2,L1,A15,C15 | Extension board |
| `SEG[6:0]` segments | K3,A14,K2,J3,H1,A16,J1 | Extension board |
| `DP` decimal point | B15 | Extension board |

# 1st Edition Design

In [56]:
%%vivado_design ring_osc
module ring_osc(
    input         sysclk,
    input  [1:0]  SW,
    input  [1:0]  btn,
    output [5:0]  HEX,
    output        DP,
    output [6:0]  SEG,
    output [1:0]  led,
    output        tx          // ← new
);

    wire rstn   = ~SW[1];
    wire en     =  SW[0];
    wire freeze =  btn[0];

    // 31-stage ring oscillator
    (* KEEP = "TRUE", DONT_TOUCH = "TRUE" *) wire [30:0] ring;

    (* KEEP = "TRUE", DONT_TOUCH = "TRUE" *)
        LUT2 #(.INIT(4'b0111)) ring_gate (
        .I0(ring[30]),
        .I1(en),
        .O (ring[0])
    );

    genvar i;
    generate
        for (i = 1; i < 31; i = i + 1) begin : ring_stages
            (* KEEP = "TRUE", DONT_TOUCH = "TRUE" *)
            LUT1 #(.INIT(2'b01)) inv (
                .I0(ring[i-1]),
                .O (ring[i])
            );
        end
    endgenerate

    wire osc_out = ring[30];

    // 2-FF CDC sampler
    (* KEEP = "TRUE" *) reg ff0, ff1;
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin ff0 <= 1'b0; ff1 <= 1'b0; end
        else       begin ff0 <= osc_out; ff1 <= ff0; end
    end

    wire entropy_bit = ff1;

    // Multi-tap entropy
    (* KEEP = "TRUE" *) wire tap7  = ring[7];
    (* KEEP = "TRUE" *) wire tap15 = ring[15];
    (* KEEP = "TRUE" *) wire tap22 = ring[22];

    wire entropy_bit_xor = ff1 ^ tap7 ^ tap15 ^ tap22;

    reg [15:0] lfsr;
    wire lfsr_feedback = lfsr[15] ^ lfsr[14] ^ lfsr[12] ^ lfsr[3]
                                  ^ entropy_bit_xor;
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) lfsr <= lfsr ^ 16'hACE1;
        else       lfsr <= {lfsr[14:0], lfsr_feedback};
    end

    function [15:0] xor_mix;
        input [15:0] x;
        reg   [15:0] h;
        begin
            h = x ^ (x >> 7);
            h = h ^ (h << 9);
            h = h ^ (h >> 13);
            xor_mix = h;
        end
    endfunction

    // Display update ~1 Hz
    reg [23:0] update_ctr;
    reg        update_tick;
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            update_ctr  <= 24'd0;
            update_tick <= 1'b0;
        end else begin
            update_tick <= (update_ctr == 24'd11999999);
            if (update_ctr == 24'd11999999) update_ctr <= 24'd0;
            else                            update_ctr <= update_ctr + 1;
        end
    end

    reg [15:0] display_word;
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn)                      display_word <= xor_mix(lfsr ^ 16'hACE1);
        else if (update_tick & ~freeze) display_word <= xor_mix(lfsr);
    end

    // 4 hex nibbles
    wire [3:0] digit3 = display_word[15:12];
    wire [3:0] digit2 = display_word[11:8];
    wire [3:0] digit1 = display_word[7:4];
    wire [3:0] digit0 = display_word[3:0];

    // 7-seg mux
    reg [13:0] mux_ctr;
    reg [1:0]  digit_sel;
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin mux_ctr <= 14'd0; digit_sel <= 2'd0; end
        else if (mux_ctr == 14'd11999) begin
            mux_ctr   <= 14'd0;
            digit_sel <= digit_sel + 1;
        end else mux_ctr <= mux_ctr + 1;
    end

    function [6:0] hex2seg;
        input [3:0] d;
        case (d)
            4'h0: hex2seg = 7'b1000000;
            4'h1: hex2seg = 7'b1111001;
            4'h2: hex2seg = 7'b0100100;
            4'h3: hex2seg = 7'b0110000;
            4'h4: hex2seg = 7'b0011001;
            4'h5: hex2seg = 7'b0010010;
            4'h6: hex2seg = 7'b0000010;
            4'h7: hex2seg = 7'b1111000;
            4'h8: hex2seg = 7'b0000000;
            4'h9: hex2seg = 7'b0010000;
            4'hA: hex2seg = 7'b0001000;
            4'hB: hex2seg = 7'b0000011;
            4'hC: hex2seg = 7'b1000110;
            4'hD: hex2seg = 7'b0100001;
            4'hE: hex2seg = 7'b0000110;
            4'hF: hex2seg = 7'b0001110;
        endcase
    endfunction

    reg [3:0] cur_nibble;
    always @(*) begin
        case (digit_sel)
            2'd0: cur_nibble = digit3;
            2'd1: cur_nibble = digit2;
            2'd2: cur_nibble = digit1;
            2'd3: cur_nibble = digit0;
        endcase
    end

    assign SEG = ~hex2seg(cur_nibble);
    assign HEX[3] = ~(digit_sel == 2'd0);
    assign HEX[2] = ~(digit_sel == 2'd1);
    assign HEX[1] = ~(digit_sel == 2'd2);
    assign HEX[0] = ~(digit_sel == 2'd3);
    assign HEX[5:4] = 2'b11;
    assign DP = 1'b0;

    // LEDs
    reg [23:0] hb_ctr;
    reg        led0_reg, led1_reg;

    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            hb_ctr   <= 24'd0;
            led0_reg <= 1'b0;
            led1_reg <= 1'b0;
        end else begin
            hb_ctr   <= hb_ctr + 1;
            led0_reg <= en;
            led1_reg <= hb_ctr[23];
        end
    end

    assign led[0] = led0_reg;
    assign led[1] = led1_reg;

    // --------------------------------------------------------
    // UART TX  — 115200 baud @ 12 MHz
    // Sends "XXXX\r\n" (4 hex chars + CR + LF) every update_tick
    // --------------------------------------------------------
    // 12000000 / 115200 = ~104 clocks per bit
    localparam CLKS_PER_BIT = 104;

    // Convert nibble to ASCII hex character
    function [7:0] nibble2ascii;
        input [3:0] n;
        nibble2ascii = (n < 4'd10) ? (8'd48 + n) : (8'd55 + n); // '0'-'9' or 'A'-'F'
    endfunction

    // TX byte buffer: "XXXX\r\n" = 6 bytes
    reg [7:0]  tx_buf [0:5];
    reg [2:0]  tx_byte_idx;   // which byte we are sending (0-5)
    reg [3:0]  tx_bit_idx;    // which bit within byte (0-9: start+8data+stop)
    reg [6:0]  tx_clk_ctr;    // clock counter per bit
    reg        tx_busy;
    reg        tx_reg;

    assign tx = tx_reg;

    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            tx_busy     <= 1'b0;
            tx_reg      <= 1'b1;  // idle high
            tx_byte_idx <= 3'd0;
            tx_bit_idx  <= 4'd0;
            tx_clk_ctr  <= 7'd0;
        end else begin

            // Load new value when update_tick fires and not busy
            if (update_tick && !tx_busy) begin
                tx_buf[0] <= nibble2ascii(display_word[15:12]);
                tx_buf[1] <= nibble2ascii(display_word[11:8]);
                tx_buf[2] <= nibble2ascii(display_word[7:4]);
                tx_buf[3] <= nibble2ascii(display_word[3:0]);
                tx_buf[4] <= 8'h0D; // CR
                tx_buf[5] <= 8'h0A; // LF
                tx_busy     <= 1'b1;
                tx_byte_idx <= 3'd0;
                tx_bit_idx  <= 4'd0;
                tx_clk_ctr  <= 7'd0;
            end

            // Transmit
            if (tx_busy) begin
                if (tx_clk_ctr < CLKS_PER_BIT - 1) begin
                    tx_clk_ctr <= tx_clk_ctr + 1;
                end else begin
                    tx_clk_ctr <= 7'd0;
                    case (tx_bit_idx)
                        4'd0: tx_reg <= 1'b0; // start bit
                        4'd1: tx_reg <= tx_buf[tx_byte_idx][0];
                        4'd2: tx_reg <= tx_buf[tx_byte_idx][1];
                        4'd3: tx_reg <= tx_buf[tx_byte_idx][2];
                        4'd4: tx_reg <= tx_buf[tx_byte_idx][3];
                        4'd5: tx_reg <= tx_buf[tx_byte_idx][4];
                        4'd6: tx_reg <= tx_buf[tx_byte_idx][5];
                        4'd7: tx_reg <= tx_buf[tx_byte_idx][6];
                        4'd8: tx_reg <= tx_buf[tx_byte_idx][7];
                        4'd9: begin
                            tx_reg <= 1'b1; // stop bit
                            if (tx_byte_idx == 3'd5) begin
                                tx_busy <= 1'b0; // done all 6 bytes
                            end else begin
                                tx_byte_idx <= tx_byte_idx + 1;
                                tx_bit_idx  <= 4'd0;
                            end
                        end
                    endcase
                    if (tx_bit_idx < 4'd9) tx_bit_idx <= tx_bit_idx + 1;
                end
            end

        end
    end

endmodule

[INFO]: Creating design file: /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/hdl/ring_osc.sv
[INFO]: Running Vivado xvlog lint:
         xvlog -sv /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/hdl/ring_osc.sv
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/hdl/ring_osc.sv" into library work
INFO: [VRFC 10-311] analyzing module ring_osc

[INFO]: Vivado xvlog lint completed with no errors (exit code 0).
[INFO]: Vivado project '/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.xpr' not found; skipping auto-add.
[INFO]: The file is saved under build/src and will be picked up when you run '%vivado prj_create'.


---

In [57]:
%%vivado_xdc A7E_ring_osc

#### On CMOD A7 35T Board ########################################################
## 12 MHz Clock
set_property -dict { PACKAGE_PIN L17   IOSTANDARD LVCMOS33 } [get_ports { sysclk }]; #IO_L12P_T1_MRCC_14 Sch=gclk
create_clock -add -name sys_clk_pin -period 83.33 -waveform {0 41.66} [get_ports {sysclk}];

## Onboard LEDs
set_property -dict { PACKAGE_PIN A17   IOSTANDARD LVCMOS33 } [get_ports { led[0] }]; #IO_L12N_T1_MRCC_16 Sch=led[1]
set_property -dict { PACKAGE_PIN C16   IOSTANDARD LVCMOS33 } [get_ports { led[1] }]; #IO_L13P_T2_MRCC_16 Sch=led[2]

## Onboard Buttons
set_property -dict { PACKAGE_PIN A18   IOSTANDARD LVCMOS33 } [get_ports { btn[0] }]; #IO_L19N_T3_VREF_16 Sch=btn[0]
set_property -dict { PACKAGE_PIN B18   IOSTANDARD LVCMOS33 } [get_ports { btn[1] }]; #IO_L19P_T3_16 Sch=btn[1]

#### On Extension Board ##########################################################
### HEX[5]
set_property -dict { PACKAGE_PIN N2    IOSTANDARD LVCMOS33 } [get_ports { HEX[5] }]; #IO_L10P_T1_AD15P_35 Sch=pio[22]
### HEX[4]
set_property -dict { PACKAGE_PIN N1    IOSTANDARD LVCMOS33 } [get_ports { HEX[4] }]; #IO_L10N_T1_AD15N_35 Sch=pio[21]
### HEX[3]
set_property -dict { PACKAGE_PIN L2    IOSTANDARD LVCMOS33 } [get_ports { HEX[3] }]; #IO_L5N_T0_AD13N_35 Sch=pio[14]
### HEX[2]
set_property -dict { PACKAGE_PIN L1    IOSTANDARD LVCMOS33 } [get_ports { HEX[2] }]; #IO_L6N_T0_VREF_35 Sch=pio[13]
### HEX[1]
set_property -dict { PACKAGE_PIN A15   IOSTANDARD LVCMOS33 } [get_ports { HEX[1] }]; #IO_L6N_T0_VREF_16 Sch=pio[07]
### HEX[0]
set_property -dict { PACKAGE_PIN C15   IOSTANDARD LVCMOS33 } [get_ports { HEX[0] }]; #IO_L11P_T1_SRCC_16 Sch=pio[05]

### DP
set_property -dict { PACKAGE_PIN B15   IOSTANDARD LVCMOS33 } [get_ports { DP }];     #IO_L11N_T1_SRCC_16 Sch=pio[08]
### SEG G
set_property -dict { PACKAGE_PIN K3    IOSTANDARD LVCMOS33 } [get_ports { SEG[6] }]; #IO_L7N_T1_AD6N_35 Sch=pio[04]
### SEG F
set_property -dict { PACKAGE_PIN A14   IOSTANDARD LVCMOS33 } [get_ports { SEG[5] }]; #IO_L6P_T0_16 Sch=pio[09]
### SEG E
set_property -dict { PACKAGE_PIN K2    IOSTANDARD LVCMOS33 } [get_ports { SEG[4] }]; #IO_L5P_T0_AD13P_35 Sch=pio[12]
### SEG D
set_property -dict { PACKAGE_PIN J3    IOSTANDARD LVCMOS33 } [get_ports { SEG[3] }]; #IO_L7P_T1_AD6P_35 Sch=pio[10]
### SEG C
set_property -dict { PACKAGE_PIN H1    IOSTANDARD LVCMOS33 } [get_ports { SEG[2] }]; #IO_L3P_T0_DQS_AD5P_35 Sch=pio[06]
### SEG B
set_property -dict { PACKAGE_PIN A16   IOSTANDARD LVCMOS33 } [get_ports { SEG[1] }]; #IO_L12P_T1_MRCC_16 Sch=pio[03]
### SEG A
set_property -dict { PACKAGE_PIN J1    IOSTANDARD LVCMOS33 } [get_ports { SEG[0] }]; #IO_L3N_T0_DQS_AD5N_35 Sch=pio[11]

### SW[1]
set_property -dict { PACKAGE_PIN W7    IOSTANDARD LVCMOS33 } [get_ports { SW[1] }];  #IO_L13P_T2_MRCC_34 Sch=pio[46]
### SW[0]
set_property -dict { PACKAGE_PIN V8    IOSTANDARD LVCMOS33 } [get_ports { SW[0] }];  #IO_L14N_T2_SRCC_34 Sch=pio[48]

set_property -dict { PACKAGE_PIN J18   IOSTANDARD LVCMOS33 } [get_ports { tx }]; #UART TX
## ============================================================
## Timing exceptions for the ring oscillator
## ============================================================
#set_false_path -from [get_nets {ring[*]}] -to [get_pins ff0_reg/D]
#set_multicycle_path -setup 2 -from [get_pins ff0_reg/Q] -to [get_pins ff1_reg/D]
#set_property DONT_TOUCH TRUE [get_nets {ring[*]}]
#set_property ALLOW_COMBINATORIAL_LOOPS TRUE [get_nets {ring[29]}]

set_property DONT_TOUCH TRUE [get_cells {ring_stages[*].inv}]
set_property DONT_TOUCH TRUE [get_cells {ring_gate}]
set_false_path -from [get_cells {ring_stages[*].inv}] -to [get_cells -hierarchical -filter {NAME =~ *ff0*}]
set_property ALLOW_COMBINATORIAL_LOOPS TRUE [get_nets {ring[29]}]

[INFO]: Creating XDC file: /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/constraints/A7E_ring_osc.xdc
[INFO]: Vivado project '/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.xpr' not found; skipping auto-add.
[INFO]: The file is saved under build/src and will be picked up when you run '%vivado prj_create'.


## Controls after programming

| Control | Location | Action |
|---|---|---|
| `SW[0]` slide right | Extension board | Starts oscillator — display rolls |
| `SW[1]` slide right | Extension board | Holds reset — display shows `00` |
| `btn[0]` hold | Onboard CMod A7 | Freezes display so you can read the value |
| `led[0]` lit | Onboard CMod A7 | Oscillator is enabled |
| `led[1]` blinking | Onboard CMod A7 | ~0.7 Hz heartbeat — board is alive |

## If digits appear on the wrong 7-seg position
The design drives `HEX[0]` for tens and `HEX[1]` for ones.
If they appear swapped or on the wrong physical digit, adjust
the `HEX` assignments at the bottom of `top_module.v` to match
your extension board's digit ordering.

## Create Vivado Project

In [58]:
%vivado prj_create

[INFO]: Running: vivado -mode batch -source /tmp/tmphpw0hha6/cmd.tcl -tclargs A7E_ring_osc xc7a35tcpg236-1 ring_osc ring_osc_tb /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/hdl /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/tb /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/constraints /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc

****** Vivado v2025.1 (64-bit)
  **** SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
  **** IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
  **** SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
  **** Start of session at: Thu Apr  2 17:22:11 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

[INFO]: Project name  : A7E_ring_osc
[INFO]: Device part   : xc7a35tcpg236-1
[INFO]: Design top    : ring_osc
[INFO]: Sim top       : ring_osc_tb
[INFO]: Project dir   : /home/user/ug2605/Dsl_notebook/DSL_

In [6]:
# not in used
%vivado sim_rtl

[ERROR]: No testbench .sv files found in '/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/tb'.


In [59]:
%vivado syn
%vivado imp

[INFO]: Running: vivado -mode batch -source /tmp/tmppdzupayn/cmd.tcl -tclargs A7E_ring_osc /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc ring_osc xc7a35tcpg236-1

****** Vivado v2025.1 (64-bit)
  **** SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
  **** IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
  **** SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
  **** Start of session at: Thu Apr  2 17:22:18 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [filemgmt 56-3] Default IP Output Path : Could not find the directory '/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.gen/sources_1'.
Scanning sources...
Finished scanning sources
[INFO]: Starting synthesis...
[Thu Apr  2 17:22:23 2026] Launched synth_1...
Run output will be captured here: /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.runs/

In [60]:
import subprocess
result = subprocess.run(
    ['grep', '-i', 'ring', 
     '/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.runs/synth_1/ring_osc.vds'],
    capture_output=True, text=True
)
print(result.stdout[:3000])

# Current directory  : /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.runs/synth_1
# Command line       : vivado -log ring_osc.vds -product Vivado -mode batch -messageDb vivado.pb -notrace -source ring_osc.tcl
# Log file           : /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.runs/synth_1/ring_osc.vds
# Journal file       : /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/A7E_ring_osc/A7E_ring_osc.runs/synth_1/vivado.jou
source ring_osc.tcl -notrace
Command: synth_design -top ring_osc -part xc7a35tcpg236-1
INFO: [Synth 8-6157] synthesizing module 'ring_osc' [/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/hdl/ring_osc.sv:4]
INFO: [Synth 8-155] case statement is not full and has no default [/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_ring_osc/src/hdl/ring_osc.sv:225]
INFO: [Synth 8-6155] done synthesizing module 'ring_osc' (0#1) [/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E